In [3]:
import pandas as pd
import numpy as np
import os

# Load your existing dataset
base = os.path.expanduser("~") + r"\phishguard-ai\datasets\phishing_urls.csv"
df = pd.read_csv(base)

print(f"Original dataset shape: {df.shape}")

# Reconstruct URL-like text strings from features
# This teaches DistilBERT to read URL patterns as text
def reconstruct_url_text(row):
    parts = []
    
    # Protocol
    if row.get('https', 0) == 1:
        parts.append("https")
    else:
        parts.append("http")
    
    # Domain characteristics as text tokens
    dots = int(row.get('qty_dot_domain', 1))
    parts.append("dot" * dots)
    
    hyphens = int(row.get('qty_hyphen_domain', 0))
    if hyphens > 0:
        parts.append("hyphen" * hyphens)
    
    # Directory depth
    slashes = int(row.get('qty_slash_directory', 0))
    if slashes > 0:
        parts.append("slash" * slashes)
    
    # Suspicious characters
    if row.get('qty_at_url', 0) > 0:
        parts.append("atsign")
    if row.get('qty_percent_url', 0) > 0:
        parts.append("percent")
    if row.get('qty_questionmark_url', 0) > 0:
        parts.append("query")
    
    # Length category
    length = row.get('length_url', 50)
    if length < 30:
        parts.append("short")
    elif length < 75:
        parts.append("medium")
    else:
        parts.append("long")
    
    # Domain age
    age = row.get('time_domain_activation', 0)
    if age < 30:
        parts.append("newdomain")
    elif age < 365:
        parts.append("youngdomain")
    else:
        parts.append("olddomain")
    
    # Shortened URL
    if row.get('url_shortened', 0) == 1:
        parts.append("shortened")
    
    return " ".join(parts)

# Apply to all rows
print("Reconstructing URL text representations...")
df['url_text'] = df.apply(reconstruct_url_text, axis=1)

print("Done.")
print(f"\nSample legitimate URL text:")
print(df[df['phishing']==0]['url_text'].iloc[0])
print(f"\nSample phishing URL text:")
print(df[df['phishing']==1]['url_text'].iloc[0])
print(f"\nLabel distribution:")
print(df['phishing'].value_counts())

Original dataset shape: (88647, 112)
Reconstructing URL text representations...
Done.

Sample legitimate URL text:
http dotdot slash short newdomain

Sample phishing URL text:
http dotdot slash short newdomain

Label distribution:
phishing
0    58000
1    30647
Name: count, dtype: int64


In [4]:
def reconstruct_url_text(row):
    parts = []
    
    # Protocol
    parts.append("https" if row.get('https', 0) == 1 else "http")
    
    # Domain dots — phishing URLs often have more subdomains
    dots = int(row.get('qty_dot_domain', 1))
    parts.append(f"domain_dots_{dots}")
    
    # Hyphens in domain — very suspicious
    hyphens = int(row.get('qty_hyphen_domain', 0))
    parts.append(f"domain_hyphens_{hyphens}")
    
    # Directory length — key feature from SHAP
    dir_len = int(row.get('directory_length', 0))
    if dir_len == 0:
        parts.append("no_directory")
    elif dir_len < 10:
        parts.append("short_directory")
    elif dir_len < 30:
        parts.append("medium_directory")
    else:
        parts.append("long_directory")
    
    # URL length
    length = int(row.get('length_url', 50))
    if length < 30:
        parts.append("short_url")
    elif length < 75:
        parts.append("medium_url")
    elif length < 120:
        parts.append("long_url")
    else:
        parts.append("very_long_url")
    
    # Slashes in URL
    slashes = int(row.get('qty_slash_url', 0))
    parts.append(f"slashes_{min(slashes, 6)}")
    
    # Suspicious characters
    if row.get('qty_at_url', 0) > 0:
        parts.append("has_at_sign")
    if row.get('qty_percent_url', 0) > 0:
        parts.append("has_percent")
    if row.get('qty_questionmark_url', 0) > 0:
        parts.append("has_query")
    if row.get('qty_dollar_directory', 0) > 0:
        parts.append("has_dollar")

    # Domain age — newly registered domains are suspicious
    age = int(row.get('time_domain_activation', 0))
    if age < 30:
        parts.append("brand_new_domain")
    elif age < 180:
        parts.append("young_domain")
    elif age < 365:
        parts.append("moderate_domain")
    else:
        parts.append("established_domain")
    
    # Domain expiration — phishing domains often expire soon
    expiry = int(row.get('time_domain_expiration', 0))
    if expiry < 180:
        parts.append("expires_soon")
    elif expiry < 365:
        parts.append("expires_year")
    else:
        parts.append("long_expiry")
    
    # TTL — low TTL is suspicious
    ttl = int(row.get('ttl_hostname', 0))
    if ttl < 300:
        parts.append("low_ttl")
    elif ttl < 3600:
        parts.append("medium_ttl")
    else:
        parts.append("high_ttl")
    
    # Redirects
    redirects = int(row.get('qty_redirects', 0))
    if redirects > 0:
        parts.append(f"redirects_{min(redirects, 5)}")
    
    # Google index
    if row.get('url_google_index', 0) == 0:
        parts.append("not_indexed")
    else:
        parts.append("google_indexed")
    
    # Shortened URL
    if row.get('url_shortened', 0) == 1:
        parts.append("shortened_url")
    
    # SPF record
    if row.get('domain_spf', 0) == 0:
        parts.append("no_spf")
    else:
        parts.append("has_spf")

    return " ".join(parts)

# Apply
print("Building richer text representations...")
df['url_text'] = df.apply(reconstruct_url_text, axis=1)

# Check how distinct they are now
legit_sample = df[df['phishing']==0]['url_text'].iloc[0]
phish_sample = df[df['phishing']==1]['url_text'].iloc[0]

print(f"\nLegitimate URL text:\n{legit_sample}")
print(f"\nPhishing URL text:\n{phish_sample}")

# Check vocabulary diversity
all_texts = df['url_text'].str.split().explode()
print(f"\nUnique tokens in vocabulary: {all_texts.nunique()}")
print(f"Most common tokens:\n{all_texts.value_counts().head(15)}")

Building richer text representations...

Legitimate URL text:
http domain_dots_2 domain_hyphens_0 short_directory short_url slashes_1 brand_new_domain expires_soon medium_ttl not_indexed no_spf

Phishing URL text:
http domain_dots_2 domain_hyphens_0 short_directory short_url slashes_1 brand_new_domain expires_soon medium_ttl not_indexed no_spf

Unique tokens in vocabulary: 70
Most common tokens:
url_text
http                  88647
not_indexed           88342
domain_hyphens_0      80570
short_directory       61321
no_spf                60882
established_domain    60005
short_url             59159
domain_dots_2         55743
slashes_0             47509
expires_soon          45780
high_ttl              31551
low_ttl               29519
has_spf               27765
medium_ttl            27577
brand_new_domain      25113
Name: count, dtype: int64


In [5]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Example URLs as text
urls = [
    "paypal secure login verify account update",
    "github repository open source python code",
    "secure paypa1 login net verify credentials bank",
]

for url in urls:
    tokens = tokenizer(url, return_tensors='pt')
    decoded = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
    print(f"\nInput:  {url}")
    print(f"Tokens: {decoded}")
    print(f"IDs:    {tokens['input_ids'][0].tolist()}")

C:\Users\tanma\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tanma\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tanma\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a


Input:  paypal secure login verify account update
Tokens: ['[CLS]', 'pay', '##pal', 'secure', 'log', '##in', 'verify', 'account', 'update', '[SEP]']
IDs:    [101, 3477, 12952, 5851, 8833, 2378, 20410, 4070, 10651, 102]

Input:  github repository open source python code
Tokens: ['[CLS]', 'gi', '##th', '##ub', 'repository', 'open', 'source', 'python', 'code', '[SEP]']
IDs:    [101, 21025, 2705, 12083, 22409, 2330, 3120, 18750, 3642, 102]

Input:  secure paypa1 login net verify credentials bank
Tokens: ['[CLS]', 'secure', 'pay', '##pa', '##1', 'log', '##in', 'net', 'verify', 'credentials', 'bank', '[SEP]']
IDs:    [101, 5851, 3477, 4502, 2487, 8833, 2378, 5658, 20410, 22496, 2924, 102]
